In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import wandb

from torch.optim import Adam
from torch.nn import BCEWithLogitsLoss
from torch.utils.data import DataLoader,TensorDataset
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

## Descargar los datos

In [2]:
!pip install kaggle
!mkdir -p data
!kaggle datasets download -d blastchar/telco-customer-churn -p data --unzip

Dataset URL: https://www.kaggle.com/datasets/blastchar/telco-customer-churn
License(s): copyright-authors
100%|█████████████████████████████████████████| 172k/172k [00:00<00:00, 571kB/s]



## Visualizar los datos

In [3]:
csv_path = '/home/jaume/ConquerX/Mis_apuntes/Pytorch/ejercicio_redes/Teleco_Customer_Churn/data/WA_Fn-UseC_-Telco-Customer-Churn.csv'

In [4]:
df = pd.read_csv(csv_path)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [5]:
df.shape

(7043, 21)

In [6]:
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [8]:
df["Churn"].value_counts()

Churn
No     5174
Yes    1869
Name: count, dtype: int64

In [9]:
# Hay que mirar columnas de tipo Object que contengan numeros siemrpe!
# Pueden tener espacios de texto vacios que isnull no contea.

df[df["TotalCharges"] == " "] # Esta es la unica columna con estas caracteristicas

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,,No
753,3115-CZMZD,Male,0,No,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,,No
936,5709-LVOEQ,Female,0,Yes,Yes,0,Yes,No,DSL,Yes,...,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,,No
1082,4367-NUYAO,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,,No
1340,1371-DWPAZ,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,,No
3331,7644-OMVMY,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,,No
3826,3213-VVOLG,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,,No
4380,2520-SGTTA,Female,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,,No
5218,2923-ARZLG,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,,No
6670,4075-WKNIU,Female,0,Yes,Yes,0,Yes,Yes,DSL,No,...,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,,No


## Interpretacion de los datos

### Contexto
"Prediga el comportamiento para fidelizar a sus clientes. Puede analizar todos los datos relevantes de los clientes y desarrollar programas de retención de clientes específicos." [Conjuntos de datos de muestra de IBM]

### Contenido
Cada fila representa a un cliente, y cada columna contiene los atributos del cliente descritos en la columna Metadatos.

### Interpretacion:
A primera vista no existen datos nulos pero si que los hay, hay columnas de tipo "Objeto" con filas vacias, por eso no los detecta como nulos.
La mayoria de las columnas son útiles para el entranmiento
La variable objetivo esta desvalanceada



## Limpieza de datos

In [10]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce") #Convierte la columna a numerica y las filas vacias en Nan

In [11]:
df = df.drop(columns=['customerID']) # Borro columnas
df = df.dropna() # Borro nulos
df = df.drop_duplicates() # Borro duplicados

In [12]:
print(df["TotalCharges"].isnull().sum()) #Comprobacion
print(df.shape)

0
(7010, 20)


In [13]:
df["Churn"] = df["Churn"].map({"No":0,"Yes":1}) # Mapear la variable objetivo

In [14]:
# Separar la variable objetivo del resto

y_dataset = df["Churn"]
x_dataset = df.drop(columns="Churn")

In [15]:
# Separar dataset TRAIN/VAL/TEST

x_train,x_test,y_train,y_test = train_test_split(
    x_dataset,y_dataset, test_size=0.2 ,train_size=0.8,random_state=1,stratify=y_dataset)

x_train,x_val,y_train,y_val = train_test_split(
    x_train,y_train, test_size=0.2 ,train_size=0.8,random_state=1,stratify=y_train)


In [16]:
# Comprobacion
print(x_train.shape)
print(x_val.shape)
print(x_test.shape)
print(y_train.shape)
print(y_val.shape)
print(y_test.shape)

(4486, 19)
(1122, 19)
(1402, 19)
(4486,)
(1122,)
(1402,)


## Transformar los datos

In [17]:
# Separar columnas categoricas y numericas para en OneHotEncoder

categorical_cols = x_train.select_dtypes(include="object").columns
numerical_cols = x_train.select_dtypes(exclude="object").columns

In [18]:
# OneHotEncoding solo a las columnas ncategoricas(para que sean numericas)
encoder = OneHotEncoder(
    handle_unknown="ignore", # Si en validation o test aparece una categoría que no existía en train,no da error.
    sparse_output=False # Devuelve un array normal de NumPy.
)

x_train_encoded = encoder.fit_transform(x_train[categorical_cols])
x_val_encoded = encoder.transform(x_val[categorical_cols])
x_test_encoded = encoder.transform(x_test[categorical_cols])


In [19]:
# Comprobacion
print(x_train_encoded.shape)
print(x_val_encoded.shape)
print(x_test_encoded.shape)

(4486, 41)
(1122, 41)
(1402, 41)


In [20]:
# Escalar las columnas numericas
scaler = StandardScaler()

x_train_num_scaled = scaler.fit_transform(x_train[numerical_cols])
x_val_num_scaled = scaler.transform(x_val[numerical_cols])
x_test_num_scaled = scaler.transform(x_test[numerical_cols])


In [21]:
# Comprobacion
print(x_train_num_scaled.shape)
print(x_val_num_scaled.shape)
print(x_test_num_scaled.shape)

(4486, 4)
(1122, 4)
(1402, 4)


## Convertir los datos en Tensores

In [22]:
# Concateno los arrays procesados para TRAIN/VAL/TEST

x_train_processed = np.concatenate(
    [x_train_num_scaled, x_train_encoded],
    axis=1
)

x_val_processed = np.concatenate(
    [x_val_num_scaled, x_val_encoded],
    axis=1
)

x_test_processed = np.concatenate(
    [x_test_num_scaled, x_test_encoded],
    axis=1
)

In [23]:
# Comprobacion
print(x_train_processed.shape)
print(x_val_processed.shape)
print(x_test_processed.shape)

(4486, 45)
(1122, 45)
(1402, 45)


In [24]:
# Combierto de array a tensor:
t_x_train = torch.from_numpy(x_train_processed).float()
t_x_val = torch.from_numpy(x_val_processed).float()
t_x_test = torch.from_numpy(x_test_processed).float()

t_y_train = torch.from_numpy(y_train.values).float().unsqueeze(1)
t_y_val = torch.from_numpy(y_val.values).float().unsqueeze(1)
t_y_test = torch.from_numpy(y_test.values).float().unsqueeze(1)


print(f"x train: {t_x_train.shape}\nx val: {t_x_val.shape}\nx test: {t_x_test.shape}\n")
print(f"y train: {t_y_train.shape}\ny val: {t_y_val.shape}\ny test: {t_y_test.shape}")

x train: torch.Size([4486, 45])
x val: torch.Size([1122, 45])
x test: torch.Size([1402, 45])

y train: torch.Size([4486, 1])
y val: torch.Size([1122, 1])
y test: torch.Size([1402, 1])


## Configuracion de parametros

In [25]:
config = {
    "input_size":t_x_train.shape[1],
    "hidden_size_1":64,
    "hidden_size_2":32,
    "batch_size":32,
    "lr":0.001,
    "epochs":3,
    "dropout":0.2,
    "weight_decay":0.0001,
    "device":"cuda",
}

In [26]:
sweep_config = {
    "name": "Telco-Customer-Churn",
    "method": "random",

    "metric": {
        "name": "val_f1",
        "goal": "maximize"
    },

    "parameters": {
        "lr": {
            "values": [0.01, 0.001, 0.0001]
        },

        "batch_size": {
            "values": [16, 32, 64]
        },

        "hidden_size_1": {
            "values": [32, 64, 128]
        },

        "hidden_size_2": {
            "values": [16, 32, 64]
        },

        "dropout": {
            "values": [0.0, 0.2, 0.4]
        },

        "weight_decay": {
            "values": [0.0, 0.0001, 0.001]
        },

        "epochs": {
            "value": 5
        }
    }
}

## Agrupar los Tensores

In [27]:
train_dataset = TensorDataset(t_x_train,t_y_train)
val_dataset = TensorDataset(t_x_val,t_y_val)
test_dataset = TensorDataset(t_x_test,t_y_test)

## Crear los DataLoaders

In [28]:
train_loader = DataLoader(train_dataset,batch_size=config["batch_size"],shuffle=True)
val_loader = DataLoader(val_dataset,batch_size=config["batch_size"],shuffle=False)
test_loader = DataLoader(test_dataset,batch_size=config["batch_size"],shuffle=False)

In [29]:
print(torch.isnan(t_x_train).sum())
print(torch.isnan(t_y_train).sum())

tensor(0)
tensor(0)


## Red Neuronal

In [30]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

In [31]:

class ChurnModel(nn.Module):

    def __init__(self, input_size, hidden_size_1, hidden_size_2, dropout):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_size_1),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_size_1, hidden_size_2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_size_2, 1)
        )

    def forward(self, x):
        return self.network(x)

## Funciones

In [32]:
def evaluate(model, data_loader, criterion, device):

    model.eval()

    total_loss = 0
    total_correct = 0
    total_samples = 0
    total_true_positives = 0
    total_false_positives = 0
    total_false_negatives = 0

    all_targets = []
    all_probabilities = []  

    with torch.no_grad():

        for x_batch, y_batch in data_loader:

            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(x_batch)

            loss = criterion(logits, y_batch)

            probabilities = torch.sigmoid(logits)

            all_targets.append(y_batch.cpu())
            
            all_probabilities.append(probabilities.cpu())  

            predictions = (probabilities >= 0.5).float()

            total_loss += loss.item() * x_batch.size(0)

            total_correct += (predictions == y_batch).sum().item()

            total_samples += y_batch.size(0)

            total_true_positives += ((predictions==1)&(y_batch==1)).sum().item()

            total_false_positives += ((predictions==1)&(y_batch==0)).sum().item()

            total_false_negatives += ((predictions==0)&(y_batch==1)).sum().item()
           
    # Calcula la precision
    if total_true_positives + total_false_positives == 0:
        precision = 0
    else:
        precision = total_true_positives / (total_true_positives + total_false_positives)

    # Calcula el racall
    if total_true_positives + total_false_negatives == 0:
        recall = 0
    else:
        recall = total_true_positives / (total_true_positives + total_false_negatives)

    # Calcula F1 Score
    if precision + recall == 0:
        f1 = 0
    else:
        f1 = 2 * (precision * recall) / (precision + recall)


    # Calcula AUC
    all_targets = torch.cat(all_targets).view(-1).numpy()
    all_probabilities = torch.cat(all_probabilities).view(-1).numpy()
    auc = roc_auc_score(all_targets, all_probabilities)
    
    average_loss = total_loss / total_samples

    accuracy = total_correct / total_samples

    return average_loss, accuracy, precision, recall, f1, auc

In [33]:
def train_model(model,train_loader,val_loader,criterion,optimizer,device,epochs):

        history = {
            "train_loss": [],
            "train_accuracy": [],
            "val_loss": [],
            "val_accuracy": [],
            "val_precision": [],
            "val_recall": [],
            "val_f1": [],
            "val_auc": []
        }

        for epoch in range(epochs):

            model.train()

            total_train_loss = 0
            total_correct = 0
            total_samples = 0

            for x_batch, y_batch in train_loader:

                x_batch = x_batch.to(device) # Carga el batch al dispositivo seleccionado
                y_batch = y_batch.to(device)

                optimizer.zero_grad() # Poner los gradientes a 0

                logits = model(x_batch) # Output del modelo en logits por falta de funcion de activacion en la ultima capa

                loss = criterion(logits, y_batch) # Calcula la perdida 

                loss.backward() 

                optimizer.step() # Actualiza los pesos usando los gradientes calculados

                probabilities = torch.sigmoid(logits) # Calcula las predicciones en probabilidades

                predictions = (probabilities >= 0.5).float() # Convierte las probabilidades en 0 o 1

                total_train_loss += loss.item() * x_batch.size(0) 

                total_correct += (predictions == y_batch).sum().item()

                total_samples += y_batch.size(0)

            train_loss = total_train_loss / total_samples

            train_accuracy = total_correct / total_samples

            val_loss, val_accuracy, val_precision, val_recall, val_f1, val_auc = evaluate(
                model,
                val_loader,
                criterion,
                device
            )

            history["train_loss"].append(train_loss)
            history["train_accuracy"].append(train_accuracy)
            history["val_loss"].append(val_loss)
            history["val_accuracy"].append(val_accuracy)
            history["val_precision"].append(val_precision)
            history["val_recall"].append(val_recall)
            history["val_f1"].append(val_f1)
            history["val_auc"].append(val_auc)

            print(
                f"Epoch {epoch + 1}/{epochs} | "
                f"Train Loss: {train_loss:.4f} | "
                f"Train Accuracy: {train_accuracy:.4f} | "
                f"Val Loss: {val_loss:.4f} | "
                f"Val Accuracy: {val_accuracy:.4f} | "
                f"Val Precision: {val_precision:.4f} | "
                f"Val Recall: {val_recall:.4f} | "
                f"Val F1: {val_f1:.4f} | "
                f"Val AUC: {val_auc:.4f} | "
            )

            wandb.log({
                "train_accuracy":train_accuracy,
                "train_loss":train_loss,
                "val_loss":val_loss,
                "val_accuracy":val_accuracy,
                "val_precision":val_precision,
                "val_recall":val_recall,
                "val_f1":val_f1,
                "val_auc":val_auc,
            })

        return history


In [40]:
def train_sweep(config_2):

    with wandb.init(
        project="Teleco-Customer-Churn",
    ) as run:
        
        config = run.config

        train_loader = DataLoader(train_dataset,batch_size=config.batch_size,shuffle=True)
        val_loader = DataLoader(val_dataset,batch_size=config.batch_size,shuffle=False)
        test_loader = DataLoader(test_dataset,batch_size=config.batch_size,shuffle=False)

        model = ChurnModel(config_2["input_size"],config.hidden_size_1,config.hidden_size_2,config.dropout).to(config_2["device"])
        criterion = BCEWithLogitsLoss()
        optimizer = Adam(
            model.parameters(),
            lr=config.lr,
            weight_decay=config.weight_decay
        )

        train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion = criterion,
            optimizer=optimizer,
            epochs= config.epochs,
            device=config_2["device"]
        )



In [35]:
model = ChurnModel(config["input_size"],config["hidden_size_1"],config["hidden_size_2"],config["dropout"])
criterion = BCEWithLogitsLoss()
optimizer = Adam(
    model.parameters(),
    lr=config["lr"],
    weight_decay=config["weight_decay"]
)

In [36]:
# train_model(
#     model=model,
#     train_loader=train_loader,
#     val_loader=val_loader,
#     criterion = criterion,
#     optimizer=optimizer,
#     epochs= config["epochs"],
#     device=config["device"]
# )

In [41]:
wandb.login()
sweep_id = wandb.sweep(sweep=sweep_config,project="Teleco-Customer-Churn")

Create sweep with ID: 58gvsae9
Sweep URL: https://wandb.ai/jpalauserrano-conquer-blocks/Teleco-Customer-Churn/sweeps/58gvsae9


In [43]:
wandb.agent(sweep_id=sweep_id, function=lambda:train_sweep(config), count=2)

wandb: Agent Starting Run: 1pnpbyxt with config:
wandb: 	batch_size: 16
wandb: 	dropout: 0.2
wandb: 	epochs: 5
wandb: 	hidden_size_1: 128
wandb: 	hidden_size_2: 32
wandb: 	lr: 0.01
wandb: 	weight_decay: 0
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/jaume/.netrc.


Epoch 1/5 | Train Loss: 0.4576 | Train Accuracy: 0.7815 | Val Loss: 0.4403 | Val Accuracy: 0.7914 | Val Precision: 0.6189 | Val Recall: 0.5522 | Val F1: 0.5836 | Val AUC: 0.8325 | 
Epoch 2/5 | Train Loss: 0.4409 | Train Accuracy: 0.7996 | Val Loss: 0.4233 | Val Accuracy: 0.8012 | Val Precision: 0.6947 | Val Recall: 0.4444 | Val F1: 0.5421 | Val AUC: 0.8405 | 
Epoch 3/5 | Train Loss: 0.4356 | Train Accuracy: 0.7938 | Val Loss: 0.4366 | Val Accuracy: 0.8066 | Val Precision: 0.6504 | Val Recall: 0.5825 | Val F1: 0.6146 | Val AUC: 0.8416 | 
Epoch 4/5 | Train Loss: 0.4377 | Train Accuracy: 0.7971 | Val Loss: 0.4214 | Val Accuracy: 0.8048 | Val Precision: 0.7120 | Val Recall: 0.4411 | Val F1: 0.5447 | Val AUC: 0.8418 | 
Epoch 5/5 | Train Loss: 0.4352 | Train Accuracy: 0.7905 | Val Loss: 0.4317 | Val Accuracy: 0.8012 | Val Precision: 0.6729 | Val Recall: 0.4848 | Val F1: 0.5636 | Val AUC: 0.8407 | 


train_accuracy,▁█▆▇▄
train_loss,█▃▁▂▁
val_accuracy,▁▆█▇▆
val_auc,▁▇██▇
val_f1,▅▁█▁▃
val_loss,█▂▇▁▅
val_precision,▁▇▃█▅
val_recall,▇▁█▁▃
train_accuracy,0.79046
train_loss,0.43521
val_accuracy,0.80125


wandb: Agent Starting Run: 5ol0w26i with config:
wandb: 	batch_size: 64
wandb: 	dropout: 0.2
wandb: 	epochs: 5
wandb: 	hidden_size_1: 32
wandb: 	hidden_size_2: 16
wandb: 	lr: 0.0001
wandb: 	weight_decay: 0.001
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/jaume/.netrc.


Epoch 1/5 | Train Loss: 0.6806 | Train Accuracy: 0.6121 | Val Loss: 0.6651 | Val Accuracy: 0.7326 | Val Precision: 0.4805 | Val Recall: 0.1246 | Val F1: 0.1979 | Val AUC: 0.7078 | 
Epoch 2/5 | Train Loss: 0.6514 | Train Accuracy: 0.7167 | Val Loss: 0.6334 | Val Accuracy: 0.7353 | Val Precision: 0.0000 | Val Recall: 0.0000 | Val F1: 0.0000 | Val AUC: 0.7497 | 
Epoch 3/5 | Train Loss: 0.6187 | Train Accuracy: 0.7379 | Val Loss: 0.5984 | Val Accuracy: 0.7353 | Val Precision: 0.0000 | Val Recall: 0.0000 | Val F1: 0.0000 | Val AUC: 0.7820 | 
Epoch 4/5 | Train Loss: 0.5846 | Train Accuracy: 0.7370 | Val Loss: 0.5627 | Val Accuracy: 0.7353 | Val Precision: 0.0000 | Val Recall: 0.0000 | Val F1: 0.0000 | Val AUC: 0.8034 | 
Epoch 5/5 | Train Loss: 0.5504 | Train Accuracy: 0.7354 | Val Loss: 0.5305 | Val Accuracy: 0.7353 | Val Precision: 0.0000 | Val Recall: 0.0000 | Val F1: 0.0000 | Val AUC: 0.8152 | 


train_accuracy,▁▇███
train_loss,█▆▅▃▁
val_accuracy,▁████
val_auc,▁▄▆▇█
val_f1,█▁▁▁▁
val_loss,█▆▅▃▁
val_precision,█▁▁▁▁
val_recall,█▁▁▁▁
train_accuracy,0.7354
train_loss,0.55041
val_accuracy,0.73529
